# Blood Bank EDA + ML Forecasting

Exploratory data analysis and demand forecasting (Exponential Smoothing + Random Forest) by blood group.

## 1️⃣ Imports & setup

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error
from statsmodels.tsa.stattools import adfuller
import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

## 2️⃣ Load dataset

In [ ]:
# Resolve path: notebook dir or backend app
_here = os.getcwd()
file_path = os.path.join(_here, "register_to_excel_updated_option2.xlsx")
if not os.path.isfile(file_path):
    file_path = os.path.join(_here, "..", "bloodbank-forecasting", "backend", "app", "register_to_excel_updated_option2.xlsx")
df = pd.read_excel(file_path)

In [ ]:
# Parse dates
df["collection_date"] = pd.to_datetime(df["Collection Date"], dayfirst=True, errors="coerce")
df["expiry_date"] = pd.to_datetime(df["Expiry Date"], dayfirst=True, errors="coerce")

# Normalize blood group
df["blood_group"] = (
    df["Blood Group"]
    .astype(str)
    .str.replace(" Pos", "+")
    .str.replace(" Neg", "-")
    .str.strip()
)
df["component"] = df["Component"].astype(str).str.upper()

print("===== DATA OVERVIEW =====")
df.info()
df.describe()

## 3️⃣ EDA — Date range & distributions

In [ ]:
print("Date range:", df["collection_date"].min(), "to", df["collection_date"].max())
print("\nBlood group distribution:")
bg_dist = df["blood_group"].value_counts()
print(bg_dist)

In [ ]:
bg_dist.plot(kind="bar", title="Blood Group Distribution")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
print("Component distribution:")
comp_dist = df["component"].value_counts()
print(comp_dist)
comp_dist.plot(kind="bar", title="Component Distribution")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4️⃣ Build daily demand (proxy)

In [ ]:
daily = (
    df.groupby(["collection_date", "blood_group"])["Quantity (ml)"]
    .sum()
    .reset_index()
)
print("Daily aggregated sample:")
daily.head()

## 5️⃣ ML forecast function (ES + Random Forest)

In [ ]:
def run_ml_for_blood_group(bg, horizon=7):
    print(f"\n=========== ML FOR {bg} ===========")

    data = daily[daily["blood_group"] == bg].sort_values("collection_date")

    if len(data) < 10:
        print("Not enough data.")
        return

    y = data["Quantity (ml)"].values

    # Stationarity check
    adf = adfuller(y)
    print("ADF p-value:", adf[1])

    split = int(len(y) * 0.8)
    train, test = y[:split], y[split:]

    # Exponential Smoothing
    model_es = ExponentialSmoothing(train, trend="add", seasonal=None).fit()
    pred_es = model_es.forecast(len(test))
    mape_es = mean_absolute_percentage_error(test, pred_es)
    rmse_es = np.sqrt(mean_squared_error(test, pred_es))
    print("\nExponential Smoothing — MAPE:", round(mape_es, 4), "RMSE:", round(rmse_es, 2))

    # Random Forest
    train_index = np.arange(len(train)).reshape(-1, 1)
    test_index = np.arange(len(train), len(y)).reshape(-1, 1)
    model_rf = RandomForestRegressor(n_estimators=100, random_state=42)
    model_rf.fit(train_index, train)
    pred_rf = model_rf.predict(test_index)
    mape_rf = mean_absolute_percentage_error(test, pred_rf)
    rmse_rf = np.sqrt(mean_squared_error(test, pred_rf))
    print("Random Forest — MAPE:", round(mape_rf, 4), "RMSE:", round(rmse_rf, 2))

    # Plot comparison
    plt.figure(figsize=(10, 5))
    plt.plot(data["collection_date"], y, label="Actual")
    plt.plot(data["collection_date"].iloc[split:], pred_es, label="ES Forecast")
    plt.plot(data["collection_date"].iloc[split:], pred_rf, label="RF Forecast")
    plt.title(f"{bg} Demand Forecast Comparison")
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    # Future 7-day forecast
    future_es = model_es.forecast(horizon)
    future_rf = model_rf.predict(np.arange(len(y), len(y) + horizon).reshape(-1, 1))
    print("\nNext 7-day (ES):", np.round(future_es, 2))
    print("Next 7-day (RF):", np.round(future_rf, 2))

## 6️⃣ Run ML for all blood groups

In [ ]:
for bg in daily["blood_group"].dropna().unique():
    run_ml_for_blood_group(bg)
print("\n=========== ANALYSIS COMPLETE ===========")